# Valencia 2024 floods from the Atlantis Hugging Face dataset

This notebook reads the **Valencia 2024** flood event (29 Oct – 4 Nov 2024) from the public
Atlantis datacube published on the Hugging Face Hub as
[`opageo/atlantis`](https://huggingface.co/datasets/opageo/atlantis). No S3 credentials
are needed.

The dataset repo has two parts:

```text
opageo/atlantis (dataset)
├── data.zarr/            # Zarr v3 datacube, one group per source
│   ├── gfm/              #   Sentinel-1 SAR (GFM)
│   ├── modis/            #   MODIS
│   └── viirs/            #   VIIRS
└── stac/                 # static STAC catalog: one collection per source, one item per date
    ├── catalog.json
    └── atlantis-datacube-<source>/<source>-<YYYY-MM-DD>/<source>-<YYYY-MM-DD>.json
```

**STAC tells you what to load; Zarr handles loading it efficiently.** The notebook:

1. inspects the dataset with the `huggingface_hub` API,
2. uses the STAC catalog to find the items that cover the Valencia event,
3. opens each item's Zarr asset over `hf://` and reads only the Valencia window,
4. summarises and plots `water_fraction` per source and day.

**Requirements:** `huggingface_hub xarray zarr>=3 fsspec pystac pandas matplotlib`, plus `cartopy` if you want a basemap.
It also works without the `atlantis` package.

> **Rate limits:** every Zarr metadata or chunk read is an HTTP request to the Hub.
> Anonymous requests are rate limited, so log in first (`hf auth login`, or set `HF_TOKEN`) to get a higher quota.

In [ ]:
%matplotlib inline

In [ ]:
from datetime import date
from pathlib import PurePosixPath

import fsspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pystac
import xarray as xr
from huggingface_hub import HfApi, HfFileSystem
from pystac.stac_io import DefaultStacIO

## 0. Configuration

The event bbox and date range match the `demo-*` / `example-valencia-gfm` targets in the repo `Makefile`.

In [ ]:
REPO_ID = "opageo/atlantis"
HF_ROOT = f"hf://datasets/{REPO_ID}"
ZARR_URL = f"{HF_ROOT}/data.zarr"
STAC_URL = f"{HF_ROOT}/stac/catalog.json"

# STAC assets still point at the original S3 store; they are remapped to ZARR_URL below.
S3_DATACUBE = "s3://atlantis/zarr/archive/datacube.zarr"

EVENT = "Valencia_2024"
BBOX = (-1.5, 38.8, 0.5, 40.0)  # west, south, east, north  (VALENCIA_BBOX)
START, END = date(2024, 10, 29), date(2024, 11, 4)  # VALENCIA_START / VALENCIA_END
SOURCES = ["gfm", "modis", "viirs"]
VARIABLE = "water_fraction"

print(f"Dataset : {REPO_ID}")
print(f"Zarr    : {ZARR_URL}")
print(f"STAC    : {STAC_URL}")
print(f"Event   : {EVENT}  bbox={BBOX}  {START} -> {END}")

## 1. Inspect the dataset with the Hub API

`HfApi` returns repo metadata. `HfFileSystem` is the fsspec filesystem behind `hf://` URLs,
so xarray, zarr and pystac can all read from the Hub through it.

In [ ]:
api = HfApi()
info = api.dataset_info(REPO_ID)
print(f"Revision (sha) : {info.sha}")
print(f"Last modified  : {info.last_modified}")

fs = HfFileSystem()
print("\nTop level      :", [PurePosixPath(p).name for p in fs.ls(f"datasets/{REPO_ID}", detail=False)])
print("data.zarr groups:", [PurePosixPath(p).name for p in fs.ls(f"datasets/{REPO_ID}/data.zarr", detail=False)])

## 2. Find the Valencia items with STAC

pystac reads the catalog through a small fsspec-backed `StacIO`, which can resolve `hf://` URLs and
the catalog's relative links.

Each collection links one item per populated date, and an item id encodes its date
(`<source>-YYYY-MM-DD`). We filter on the link hrefs first and only fetch the few item JSONs
inside the event window. Walking thousands of items one by one would quickly hit the Hub's rate limits.

In [ ]:
class FsspecStacIO(DefaultStacIO):
    def read_text_from_href(self, href: str) -> str:
        with fsspec.open(href, "rt", encoding="utf-8") as f:
            return f.read()


catalog = pystac.Catalog.from_file(STAC_URL, stac_io=FsspecStacIO())
collections = {c.id.removeprefix("atlantis-datacube-"): c for c in catalog.get_children()}
print(f"Catalog     : {catalog.id} - {catalog.title}")
print(f"Collections : {list(collections)}")

In [ ]:
def bbox_intersects(a, b) -> bool:
    return a[0] < b[2] and b[0] < a[2] and a[1] < b[3] and b[1] < a[3]


def event_items(collection: pystac.Collection) -> list[pystac.Item]:
    """Items of *collection* dated within [START, END] whose footprint intersects BBOX."""
    items = []
    for link in collection.get_links("item"):
        day = date.fromisoformat(PurePosixPath(link.get_href()).stem.split("-", 1)[1])
        if not START <= day <= END:
            continue
        item = link.resolve_stac_object(root=catalog).target
        if item.bbox and bbox_intersects(item.bbox, BBOX):
            items.append(item)
    return items


items_by_source = {s: event_items(collections[s]) for s in SOURCES if s in collections}
for source, items in items_by_source.items():
    print(f"{source:6s}: {len(items)} item(s) -> {[it.id for it in items]}")

In [ ]:
example = next(it for items in items_by_source.values() for it in items)
asset = example.assets["zarr"]
print(f"Item            : {example.id}  datetime={example.datetime}")
print(f"Variables       : {list(example.properties.get('cube:variables', {}))}")
print(f"Asset href      : {asset.href}")
print(f"xarray kwargs   : {asset.extra_fields.get('xarray:open_kwargs')}")

## 3. Open the Zarr asset from the Hub and read the Valencia window

Each item's `zarr` asset points at the shared datacube, and its `xarray:open_kwargs` select the
source group. We swap the S3 href for the `hf://` copy and open the store lazily. Only the chunks that
intersect the Valencia bbox and the item dates are downloaded.

The cube is on a global 1-arcmin EPSG:4326 grid, and `y` runs north to south, so the latitude slice is `(north, south)`.

In [ ]:
def open_item_cube(item: pystac.Item) -> xr.Dataset:
    asset = item.assets["zarr"]
    href = asset.href.replace(S3_DATACUBE, ZARR_URL)
    kwargs = {k: v for k, v in asset.extra_fields["xarray:open_kwargs"].items() if k != "engine"}
    kwargs["consolidated"] = None  # use consolidated metadata when present, else per-array metadata
    return xr.open_zarr(href, **kwargs)


west, south, east, north = BBOX
cubes: dict[str, xr.DataArray] = {}
for source, items in items_by_source.items():
    if not items:
        continue
    ds = open_item_cube(items[0])  # all items of a source share the same store/group
    times = [np.datetime64(it.datetime.replace(tzinfo=None), "ns") for it in items]
    da = ds[VARIABLE].sel(x=slice(west, east), y=slice(north, south)).sel(time=times)
    cubes[source] = da.load()
    print(f"{source:6s}: {dict(da.sizes)}  ({da.nbytes / 1e6:.2f} MB decoded)")

## 4. Per-day summary

`water_fraction` is stored as `uint8` in `[0, 100]` with fill value 255. CF decoding turns it into a float in `[0, 1]`, with `NaN` where there is no observation.

In [ ]:
rows = []
for source, da in cubes.items():
    valid = da.notnull().sum(["y", "x"])
    flooded = (da > 0.5).sum(["y", "x"])
    for t in da["time"].values:
        n_valid = int(valid.sel(time=t))
        rows.append({
            "source": source,
            "date": np.datetime_as_string(t, unit="D"),
            "valid_px": n_valid,
            "flooded_px (>0.5)": int(flooded.sel(time=t)),
            "mean_water_fraction": float(da.sel(time=t).mean()) if n_valid else np.nan,
        })

summary = pd.DataFrame(rows)
summary.pivot(index="date", columns="source", values="flooded_px (>0.5)")

In [ ]:
summary

## 5. Compare the sources side by side

On each day with valid GFM (Sentinel-1) pixels over the AOI, the three sources are plotted next to each other, all using the same `[0, 1]` colour scale. If MODIS or VIIRS has nothing on that day, its panel is marked "no observation". If `cartopy` is installed, coastlines and borders are drawn on top.

In [ ]:
try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
except ImportError:
    ccrs = None
    print("cartopy not installed - plotting without a basemap.")

PANEL_HEIGHT = 3.5
# Match each panel's aspect to the AOI so cartopy's equal-aspect axes aren't letterboxed.
PANEL_WIDTH = PANEL_HEIGHT * (east - west) / (north - south)


def by_day(da: xr.DataArray) -> dict[str, xr.DataArray]:
    return {np.datetime_as_string(t, unit="D"): da.sel(time=t) for t in da["time"].values}


def has_data(da: xr.DataArray | None) -> bool:
    return da is not None and bool(da.notnull().any())


def plot_sources(day: str, daily: dict[str, dict[str, xr.DataArray]]) -> None:
    """Plot every source side by side for *day*, cropped to the AOI, with one shared colorbar."""
    subplot_kw = {"projection": ccrs.PlateCarree()} if ccrs is not None else {}
    fig, axes = plt.subplots(
        1,
        len(SOURCES),
        figsize=(PANEL_WIDTH * len(SOURCES) + 1.0, PANEL_HEIGHT),
        subplot_kw=subplot_kw,
        constrained_layout=True,
    )
    mappable = None
    for i, (ax, source) in enumerate(zip(axes, SOURCES)):
        da_day = daily.get(source, {}).get(day)
        if has_data(da_day):
            transform = {"transform": ccrs.PlateCarree()} if ccrs is not None else {}
            mappable = da_day.plot(ax=ax, cmap="Blues", vmin=0.0, vmax=1.0, add_colorbar=False, **transform)
        else:
            ax.text(0.5, 0.5, "no observation", transform=ax.transAxes, ha="center", va="center", color="grey")

        if ccrs is not None:
            ax.set_extent([west, east, south, north], crs=ccrs.PlateCarree())
            ax.add_feature(cfeature.COASTLINE.with_scale("10m"), linewidth=0.6)
            ax.add_feature(cfeature.BORDERS.with_scale("10m"), linewidth=0.4, linestyle=":")
            ax.gridlines(draw_labels=["bottom", "left"] if i == 0 else ["bottom"], linewidth=0.3, alpha=0.5)
        else:
            ax.set_xlim(west, east)
            ax.set_ylim(south, north)
            ax.set_xlabel("Longitude (°)")
            ax.set_ylabel("Latitude (°)" if i == 0 else "")
        ax.set_title(source.upper(), fontsize=10, fontweight="bold")

    fig.colorbar(mappable, ax=axes, label=VARIABLE, shrink=0.9, pad=0.02)
    fig.suptitle(f"{EVENT}: {VARIABLE} {day}", fontweight="bold")
    plt.show()
    plt.close(fig)


daily = {source: by_day(da) for source, da in cubes.items()}
gfm_days = sorted(day for day, da_day in daily.get("gfm", {}).items() if has_data(da_day))
print(f"Days with valid GFM observations over the AOI: {gfm_days}")

for day in gfm_days:
    plot_sources(day, daily)

## 6. Direct access without STAC

If you already know the source and dates, you can skip STAC and open a group directly.
You can also pin an exact dataset revision for reproducibility by putting it in the URL:
`hf://datasets/opageo/atlantis@<sha>/data.zarr`.

In [ ]:
ds_direct = xr.open_zarr(ZARR_URL, group="viirs", consolidated=None)
valencia = ds_direct[VARIABLE].sel(
    x=slice(west, east),
    y=slice(north, south),
    time=slice(np.datetime64(START), np.datetime64(END)),
)
print(f"Full cube : {dict(ds_direct.sizes)}")
print(f"Valencia  : {dict(valencia.sizes)}  (lazy - nothing downloaded until .load()/.values)")